In [1]:
GITHUB_REPO_URL = "https://github.com/24520152-ben/LLM-Generated-Multi-Level-Adversarial-Attacks-on-Vietnamese-Hate-Speech-Detection"
BASELINE_OPTIONS = {
    "model": "phobert",
    "epochs": 10,
    "device": "cuda:0",
}  # Use cuda:1 for the second GPU.

In [2]:
import json
import os
import subprocess
from pathlib import Path

kaggle_working = Path("/kaggle/working")
project_root = Path.cwd()
if kaggle_working.is_dir():
    project_root = kaggle_working / "ie403"
    if not project_root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--", GITHUB_REPO_URL, str(project_root)],
            check=True,
        )
elif not (project_root / "src" / "vihsd_attack").is_dir():
    project_root = project_root.parent

source_dir = project_root / "src"
if not (source_dir / "vihsd_attack").is_dir():
    raise RuntimeError(f"Cannot find src/vihsd_attack in {project_root}.")
env = {
    **os.environ,
    "UV_PROJECT_ENVIRONMENT": str(project_root / ".venv"),
    "PYTHONPATH": str(source_dir),
}
if kaggle_working.is_dir() and not env.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient

        env["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass

subprocess.run(
    ["uv", "sync", "--locked", "--python", "3.12"],
    cwd=project_root,
    env=env,
    check=True,
)

train_code = """
import json
import sys
from vihsd_attack.classifiers.service import run_baseline
from vihsd_attack.classifiers.config import BaselineConfig

run_baseline(BaselineConfig(**json.loads(sys.argv[1])))
"""
subprocess.run(
    [
        "uv",
        "run",
        "--no-sync",
        "python",
        "-u",
        "-c",
        train_code,
        json.dumps(BASELINE_OPTIONS, default=str),
    ],
    cwd=project_root,
    env=env,
    check=True,
)

Resolved 82 packages in 1ms
Checked 79 packages in 1ms
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 16899.03it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING

Training phobert on cuda:0


phobert epoch 1/10: 100%|██████████| 1503/1503 [04:14<00:00,  5.90it/s]


Epoch 1: validation macro-F1 = 60.77%


phobert epoch 2/10: 100%|██████████| 1503/1503 [04:20<00:00,  5.76it/s]


Epoch 2: validation macro-F1 = 66.91%


... (15 outputs left out)

Epoch 10: validation macro-F1 = 67.25%


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2732.59it/s]


test: macro-F1 = 64.10%, accuracy = 87.53%
Saved baseline: /kaggle/working/artifacts/baselines/phobert/seed-42_20261125T164416Z_1f3bb9b3


CompletedProcess(args=['uv', 'run', '--no-sync', 'python', '-u', '-c', '\nimport json\nimport sys\nfrom vihsd_attack.classifiers.service import run_baseline\nfrom vihsd_attack.config import BaselineConfig\n\nrun_baseline(BaselineConfig(**json.loads(sys.argv[1])))\n', '{"model": "phobert", "epochs": 10, "device": "cuda:0"}'], returncode=0)